# Hybrid MPI-CUDA & Cache-Line Coherency Lab
**Student:** Ainat Yermurat — **ID:** 230103161  
**Instructor:** Sufyan bin Uzayr  
Runtime: Google Colab, T4 GPU, 4 MPI ranks (2×2 Cartesian grid).

## Cell 1 — Hardware & MPI environment setup

In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!lscpu | grep -E "Model name|CPU\(s\)|cache" || true
!cat /sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size || true
!apt-get -qq update > /dev/null && apt-get -qq install -y openmpi-bin libopenmpi-dev > /dev/null
!pip install -q --no-binary=mpi4py mpi4py
!pip install -q numba numpy
!mpirun --version | head -n 1
!python3 -c "import numba, mpi4py; print('numba', numba.__version__, '| mpi4py', mpi4py.__version__)"


name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-501dd0b0-086c-7321-b66a-8e2cf85952c5, 7.5, 15360 MiB
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
L1d cache:                               32 KiB (1 instance)
L1i cache:                               32 KiB (1 instance)
L2 cache:                                1 MiB (1 instance)
L3 cache:                                38.5 MiB (1 instance)
NUMA node0 CPU(s):                       0,1
64
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
mpirun (Open MPI) 4.1.6
numba 0.64.0 | mpi4py 4.1.2


## Cell 2 — Solo student identity configuration

In [3]:
STUDENT_NAME = "Ainat Yermurat"
STUDENT_ID = "230103161"
GITHUB_USERNAME = "yermy-bc"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "Set GITHUB_USERNAME to your real GitHub handle before running."
)
print(f"Verified solo student handle: {GITHUB_USERNAME} | {STUDENT_NAME} ({STUDENT_ID})")


Verified solo student handle: yermy-bc | Ainat Yermurat (230103161)


## Cell 3 — Hybrid MPI-CUDA & shared-memory coherency engine

In [4]:
%%writefile hybrid_mpi_cuda_cache.py
"""
Hybrid MPI-CUDA & Cache-Line Coherency Lab
Student : Ainat Yermurat (ID 230103161)
Instructor: Sufyan bin Uzayr

Run:  mpirun --allow-run-as-root --oversubscribe -np 4 python3 hybrid_mpi_cuda_cache.py <github_username>
"""
import os
import sys
import json
import math
import time
import hashlib
import platform
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, njit, float64, uint64
from numba.core import types, cgutils
from numba.core.extending import intrinsic

STUDENT_NAME = "Ainat Yermurat"
STUDENT_ID = "230103161"

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2  # 18x18 shared tile
GLOBAL_NX = 512
GLOBAL_NY = 512
LOCAL_N = 256              # interior points per rank per dimension
PAD_N = LOCAL_N + 2        # 258 with ghost layer
STENCIL_STEPS = 300
SHM_ITERS = 40_000_000
LAMBDA_DAMP = 0.15
SHM_SLOTS = 64
CFL_SAFETY = 0.8           # dt = 0.8 * dt_max
ENERGY_STEPS = (0, 100, 200, 300)

# message tags = direction the payload travels
TAG_UP, TAG_DOWN, TAG_LEFT, TAG_RIGHT = 11, 12, 13, 14


def get_gpu_uuid() -> str:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def get_gpu_name() -> str:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            encoding="utf-8",
        )
        return out.strip().splitlines()[0].strip()
    except Exception:
        return "unknown"


def get_cpu_model() -> str:
    try:
        with open("/proc/cpuinfo", "r", encoding="utf-8") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "unknown"


def detect_cpu_line_size() -> int:
    path = "/sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size"
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return int(f.read().strip())
    return 64


def log0(ctx, msg):
    if ctx["rank"] == 0:
        print(msg, flush=True)


# ==============================================================================
# TASK 1: Hybrid MPI Cartesian Grid + Node-Local MPI-3 Shared-Memory Window
# ==============================================================================
def init_hybrid_context(github_user: str) -> dict:
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Requires exactly 4 MPI ranks (2x2 grid), got {size}.")

    # --- Cartesian 2x2, non-periodic -------------------------------------
    cart_comm = comm.Create_cart(dims=[2, 2], periods=[False, False], reorder=False)
    cart_rank = cart_comm.Get_rank()
    coords = cart_comm.Get_coords(cart_rank)           # (row, col)
    up, down = cart_comm.Shift(0, 1)                   # row-1 / row+1
    left, right = cart_comm.Shift(1, 1)                # col-1 / col+1

    # --- GPU binding (single T4 shared by all 4 ranks) --------------------
    cuda.select_device(0)

    # --- Deterministic instance seed --------------------------------------
    gpu_uuid = comm.bcast(get_gpu_uuid() if rank == 0 else None, root=0)
    cpu_sig = f"{platform.machine()}::{os.cpu_count()}::{get_cpu_model()}"
    cpu_sig = comm.bcast(cpu_sig if rank == 0 else None, root=0)
    line_bytes = detect_cpu_line_size()
    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}::{cpu_sig}::{line_bytes}".encode("utf-8")
    ).hexdigest()
    instance_seed = int(digest[:16], 16)  # 64-bit

    # --- Node-local shared memory window ----------------------------------
    node_comm = comm.Split_type(MPI.COMM_TYPE_SHARED)
    node_rank = node_comm.Get_rank()
    itemsize = 8
    # over-allocate by one cache line so we can align the view to 64 bytes
    nbytes = (SHM_SLOTS * itemsize + 64) if node_rank == 0 else 0
    win = MPI.Win.Allocate_shared(nbytes, itemsize, comm=node_comm)
    buf, qitemsize = win.Shared_query(0)
    raw = np.ndarray(buffer=buf, dtype=np.uint64, shape=(SHM_SLOTS + 8,))
    base_addr = raw.ctypes.data
    # mmap'd segments are page-aligned in every process, so the offset is identical;
    # compute it once on node rank 0 and broadcast to be safe.
    offset_words = ((64 - (base_addr % 64)) % 64) // itemsize
    offset_words = node_comm.bcast(offset_words if node_rank == 0 else None, root=0)
    shm_view = raw[offset_words: offset_words + SHM_SLOTS]
    aligned_addr = shm_view.ctypes.data
    assert aligned_addr % 64 == 0, "shared window view is not 64-byte aligned"
    if node_rank == 0:
        shm_view[:] = 0
    node_comm.Barrier()

    ctx = dict(
        comm=comm, rank=rank, size=size,
        cart_comm=cart_comm, cart_rank=cart_rank, coords=tuple(coords),
        up=up, down=down, left=left, right=right,
        node_comm=node_comm, node_rank=node_rank, node_size=node_comm.Get_size(),
        win=win, shm_view=shm_view, shm_base_addr=base_addr,
        shm_aligned_addr=aligned_addr, shm_offset_words=offset_words,
        gpu_uuid=gpu_uuid, cpu_sig=cpu_sig, line_bytes=line_bytes,
        instance_seed=instance_seed, github_user=github_user.strip(),
    )
    return ctx


# ==============================================================================
# TASK 2: Rank-local fields + halo pack/unpack kernels
# ==============================================================================
def initialize_fields(ctx: dict) -> dict:
    """Build rank-local (258,258) arrays from global coordinates only."""
    row, col = ctx["coords"]
    h = 1.0 / (GLOBAL_NX + 1)                       # dx = 1/513
    rng = np.random.default_rng(ctx["instance_seed"])
    # seed-dependent (but rank-identical) Gaussian bump parameters
    x0 = 0.35 + 0.30 * rng.random()
    y0 = 0.35 + 0.30 * rng.random()
    amp = 0.8 + 0.4 * rng.random()
    sigma = 0.08 + 0.04 * rng.random()
    phase = 2.0 * math.pi * rng.random()

    # local index i (0..257) -> global index gi = row*256 + i (0..513)
    gi = row * LOCAL_N + np.arange(PAD_N, dtype=np.float64)
    gj = col * LOCAL_N + np.arange(PAD_N, dtype=np.float64)
    Y, X = np.meshgrid(gi * h, gj * h, indexing="ij")

    kappa = 0.5 + 0.4 * np.sin(2.0 * math.pi * X + phase) * np.cos(2.0 * math.pi * Y)
    source = 0.5 * np.sin(math.pi * X) * np.sin(math.pi * Y)
    u = amp * np.exp(-((X - x0) ** 2 + (Y - y0) ** 2) / (2.0 * sigma ** 2))

    # Dirichlet u = 0 on the physical boundary (global index 0 and 513),
    # and ghost cells start at zero (filled later by halo exchange).
    u[0, :] = 0.0
    u[-1, :] = 0.0
    u[:, 0] = 0.0
    u[:, -1] = 0.0
    source[0, :] = source[-1, :] = 0.0
    source[:, 0] = source[:, -1] = 0.0

    u = np.ascontiguousarray(u)
    kappa = np.ascontiguousarray(kappa)
    source = np.ascontiguousarray(source)

    local_kmax = float(kappa[1:-1, 1:-1].max())
    kappa_max = ctx["cart_comm"].allreduce(local_kmax, op=MPI.MAX)
    dt_max = h * h / (4.0 * kappa_max)
    dt = CFL_SAFETY * dt_max

    return dict(
        u0=u, kappa=kappa, source=source, h=h, dt=dt, dt_max=dt_max,
        kappa_max=kappa_max,
        init_params=dict(x0=x0, y0=y0, amp=amp, sigma=sigma, phase=phase),
    )


@cuda.jit
def pack_halos_kernel(u, send_buf):
    """send_buf layout (4*256): [top row | bottom row | left col | right col]."""
    t = cuda.grid(1)
    if t < 4 * LOCAL_N:
        side = t // LOCAL_N
        k = t % LOCAL_N + 1
        if side == 0:
            send_buf[t] = u[1, k]               # contiguous row 1
        elif side == 1:
            send_buf[t] = u[LOCAL_N, k]         # contiguous row 256
        elif side == 2:
            send_buf[t] = u[k, 1]               # strided column 1
        else:
            send_buf[t] = u[k, LOCAL_N]         # strided column 256


@cuda.jit
def unpack_halos_kernel(u, recv_buf, has_up, has_down, has_left, has_right):
    """recv_buf layout: [from up -> row 0 | from down -> row 257 |
                         from left -> col 0 | from right -> col 257]."""
    t = cuda.grid(1)
    if t < 4 * LOCAL_N:
        side = t // LOCAL_N
        k = t % LOCAL_N + 1
        if side == 0:
            if has_up:
                u[0, k] = recv_buf[t]
        elif side == 1:
            if has_down:
                u[LOCAL_N + 1, k] = recv_buf[t]
        elif side == 2:
            if has_left:
                u[k, 0] = recv_buf[t]
        else:
            if has_right:
                u[k, LOCAL_N + 1] = recv_buf[t]


# ==============================================================================
# TASK 3: 16x16 threadblock stencil with 18x18 shared tile + dual streams
# ==============================================================================
STENCIL_SIG = ("void(float64[:, ::1], float64[:, ::1], float64[:, ::1], float64[:, ::1], "
               "float64, float64, float64, int64, int64, int64, int64)")


@cuda.jit(STENCIL_SIG)
def tiled_stencil_kernel(u, kappa, source, u_new, dt, inv_h2, lam, i0, i1, j0, j1):
    """Updates u_new[i,j] for i in [i0,i1], j in [j0,j1] (inclusive).
    u_t = div(kappa grad u) - lam*u + S, explicit Euler, face kappa = arithmetic mean."""
    s_u = cuda.shared.array(shape=(SHARED_DIM, SHARED_DIM), dtype=float64)
    s_k = cuda.shared.array(shape=(SHARED_DIM, SHARED_DIM), dtype=float64)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    base_i = i0 + cuda.blockIdx.y * TILE_DIM
    base_j = j0 + cuda.blockIdx.x * TILE_DIM
    tid = ty * TILE_DIM + tx

    # cooperative 18x18 load (324 cells by 256 threads), restricted to region +/- 1
    idx = tid
    while idx < SHARED_DIM * SHARED_DIM:
        si = idx // SHARED_DIM
        sj = idx % SHARED_DIM
        gi = base_i - 1 + si
        gj = base_j - 1 + sj
        if gi >= i0 - 1 and gi <= i1 + 1 and gj >= j0 - 1 and gj <= j1 + 1:
            s_u[si, sj] = u[gi, gj]
            s_k[si, sj] = kappa[gi, gj]
        else:
            s_u[si, sj] = 0.0
            s_k[si, sj] = 0.0
        idx += TILE_DIM * TILE_DIM

    cuda.syncthreads()

    i = base_i + ty
    j = base_j + tx
    if i <= i1 and j <= j1:
        si = ty + 1
        sj = tx + 1
        uc = s_u[si, sj]
        kc = s_k[si, sj]
        k_n = 0.5 * (kc + s_k[si - 1, sj])
        k_s = 0.5 * (kc + s_k[si + 1, sj])
        k_w = 0.5 * (kc + s_k[si, sj - 1])
        k_e = 0.5 * (kc + s_k[si, sj + 1])
        flux = (k_e * (s_u[si, sj + 1] - uc) - k_w * (uc - s_u[si, sj - 1])
                + k_s * (s_u[si + 1, sj] - uc) - k_n * (uc - s_u[si - 1, sj]))
        u_new[i, j] = uc + dt * (flux * inv_h2 - lam * uc + source[i, j])


def launch_stencil(region, stream, d_u, d_k, d_s, d_unew, dt, inv_h2):
    i0, i1, j0, j1 = region
    ni = i1 - i0 + 1
    nj = j1 - j0 + 1
    grid = ((nj + TILE_DIM - 1) // TILE_DIM, (ni + TILE_DIM - 1) // TILE_DIM)
    block = (TILE_DIM, TILE_DIM)
    tiled_stencil_kernel[grid, block, stream](
        d_u, d_k, d_s, d_unew, dt, inv_h2, LAMBDA_DAMP, i0, i1, j0, j1)


FULL_REGION = (1, LOCAL_N, 1, LOCAL_N)
DEEP_REGION = (2, LOCAL_N - 1, 2, LOCAL_N - 1)            # [2..255, 2..255]
BORDER_REGIONS = (
    (1, 1, 1, LOCAL_N),                                    # top row
    (LOCAL_N, LOCAL_N, 1, LOCAL_N),                        # bottom row
    (2, LOCAL_N - 1, 1, 1),                                # left column
    (2, LOCAL_N - 1, LOCAL_N, LOCAL_N),                    # right column
)
HALO_THREADS = 256
HALO_BLOCKS = (4 * LOCAL_N + HALO_THREADS - 1) // HALO_THREADS


class GPUState:
    def __init__(self, ctx, fields):
        self.ctx = ctx
        self.f = fields
        self.inv_h2 = 1.0 / (fields["h"] ** 2)
        self.dt = fields["dt"]
        self.d_k = cuda.to_device(fields["kappa"])
        self.d_s = cuda.to_device(fields["source"])
        self.d_u = cuda.to_device(fields["u0"])
        self.d_unew = cuda.to_device(fields["u0"])
        self.d_send = cuda.device_array(4 * LOCAL_N, dtype=np.float64)
        self.d_recv = cuda.device_array(4 * LOCAL_N, dtype=np.float64)
        self.h_send = cuda.pinned_array(4 * LOCAL_N, dtype=np.float64)
        self.h_recv = cuda.pinned_array(4 * LOCAL_N, dtype=np.float64)
        self.h_send[:] = 0.0
        self.h_recv[:] = 0.0
        self.stream_interior = cuda.stream()
        self.stream_boundary = cuda.stream()
        c = ctx
        self.flags = (int(c["up"] != MPI.PROC_NULL), int(c["down"] != MPI.PROC_NULL),
                      int(c["left"] != MPI.PROC_NULL), int(c["right"] != MPI.PROC_NULL))

    def reset(self):
        self.d_u.copy_to_device(self.f["u0"])
        self.d_unew.copy_to_device(self.f["u0"])
        cuda.synchronize()

    def swap(self):
        self.d_u, self.d_unew = self.d_unew, self.d_u

    def exchange_host(self):
        """Non-blocking tag-matched MPI exchange over pinned host buffers."""
        c = self.ctx
        cart = c["cart_comm"]
        n = LOCAL_N
        hs, hr = self.h_send, self.h_recv
        reqs = [
            cart.Irecv([hr[0 * n:1 * n], MPI.DOUBLE], source=c["up"], tag=TAG_DOWN),
            cart.Irecv([hr[1 * n:2 * n], MPI.DOUBLE], source=c["down"], tag=TAG_UP),
            cart.Irecv([hr[2 * n:3 * n], MPI.DOUBLE], source=c["left"], tag=TAG_RIGHT),
            cart.Irecv([hr[3 * n:4 * n], MPI.DOUBLE], source=c["right"], tag=TAG_LEFT),
            cart.Isend([hs[0 * n:1 * n], MPI.DOUBLE], dest=c["up"], tag=TAG_UP),
            cart.Isend([hs[1 * n:2 * n], MPI.DOUBLE], dest=c["down"], tag=TAG_DOWN),
            cart.Isend([hs[2 * n:3 * n], MPI.DOUBLE], dest=c["left"], tag=TAG_LEFT),
            cart.Isend([hs[3 * n:4 * n], MPI.DOUBLE], dest=c["right"], tag=TAG_RIGHT),
        ]
        return reqs

    def step_serialized(self):
        sb = self.stream_boundary
        pack_halos_kernel[HALO_BLOCKS, HALO_THREADS, sb](self.d_u, self.d_send)
        self.d_send.copy_to_host(self.h_send, stream=sb)
        sb.synchronize()
        MPI.Request.Waitall(self.exchange_host())
        self.d_recv.copy_to_device(self.h_recv, stream=sb)
        unpack_halos_kernel[HALO_BLOCKS, HALO_THREADS, sb](self.d_u, self.d_recv, *self.flags)
        launch_stencil(FULL_REGION, sb, self.d_u, self.d_k, self.d_s, self.d_unew,
                       self.dt, self.inv_h2)
        sb.synchronize()
        self.swap()

    def step_overlapped(self):
        si, sb = self.stream_interior, self.stream_boundary
        # 1) boundary stream: pack strided/contiguous edges + async D2H into pinned memory
        pack_halos_kernel[HALO_BLOCKS, HALO_THREADS, sb](self.d_u, self.d_send)
        self.d_send.copy_to_host(self.h_send, stream=sb)
        # 2) interior stream: deep interior [2..255]^2 needs no ghost data
        launch_stencil(DEEP_REGION, si, self.d_u, self.d_k, self.d_s, self.d_unew,
                       self.dt, self.inv_h2)
        # 3) MPI exchange while the interior kernel runs
        sb.synchronize()
        MPI.Request.Waitall(self.exchange_host())
        # 4) H2D + unpack ghosts + 4 border strips on the boundary stream
        self.d_recv.copy_to_device(self.h_recv, stream=sb)
        unpack_halos_kernel[HALO_BLOCKS, HALO_THREADS, sb](self.d_u, self.d_recv, *self.flags)
        for reg in BORDER_REGIONS:
            launch_stencil(reg, sb, self.d_u, self.d_k, self.d_s, self.d_unew,
                           self.dt, self.inv_h2)
        sb.synchronize()
        si.synchronize()
        self.swap()

    def global_energy(self):
        u = self.d_u.copy_to_host()
        h = self.f["h"]
        local = 0.5 * h * h * float(np.sum(u[1:-1, 1:-1] ** 2))
        return self.ctx["cart_comm"].allreduce(local, op=MPI.SUM)


def run_mode(state: GPUState, mode: str, steps: int):
    state.reset()
    ctx = state.ctx
    step_fn = state.step_overlapped if mode == "overlapped" else state.step_serialized
    energies = {}
    compute_ms = 0.0
    if 0 in ENERGY_STEPS:
        energies[0] = state.global_energy()
    ctx["cart_comm"].Barrier()
    for s in range(1, steps + 1):
        t0 = time.perf_counter()
        step_fn()
        compute_ms += (time.perf_counter() - t0) * 1e3
        if s in ENERGY_STEPS:
            energies[s] = state.global_energy()
    cuda.synchronize()
    ctx["cart_comm"].Barrier()
    total_ms = ctx["cart_comm"].allreduce(compute_ms, op=MPI.MAX)
    final = state.d_u.copy_to_host()
    return final, energies, total_ms


# ==============================================================================
# TASK 4: Host false-sharing sweep on the MPI-3 shared window
# ==============================================================================
@intrinsic
def _relaxed_rmw_add(typingctx, arr, idx, inc):
    """In-place RMW through memory: monotonic (relaxed) atomic load + add + monotonic
    atomic store. Not a locked RMW, but LLVM may NOT promote it to a register
    (LICM only promotes simple loads/stores), so every iteration touches the line."""
    if not isinstance(arr, types.Array) or arr.dtype != types.uint64:
        return None
    sig = types.uint64(arr, idx, inc)

    def codegen(context, builder, signature, args):
        aryty, idxty, incty = signature.args
        ary = context.make_array(aryty)(context, builder, args[0])
        index = context.cast(builder, args[1], idxty, types.intp)
        ptr = cgutils.get_item_pointer(context, builder, aryty, ary, [index],
                                       wraparound=False)
        delta = context.cast(builder, args[2], incty, types.uint64)
        old = builder.load_atomic(ptr, "monotonic", 8)
        new = builder.add(old, delta)
        builder.store_atomic(new, ptr, "monotonic", 8)
        return new

    return sig, codegen


@njit(nogil=True)
def host_shm_accumulate(shm_view, slot, iters):
    one = np.uint64(1)
    last = np.uint64(0)
    for _ in range(iters):
        last = _relaxed_rmw_add(shm_view, slot, one)
    return last


def benchmark_shm_strides(ctx: dict) -> dict:
    node_comm = ctx["node_comm"]
    view = ctx["shm_view"]
    rank = ctx["node_rank"]
    nsz = ctx["node_size"]

    # JIT warm-up on a private slot pattern (not timed)
    host_shm_accumulate(view, rank * 16, 1000)
    node_comm.Barrier()

    results = {}
    for stride in (1, 8, 16):
        if rank == 0:
            view[:] = 0
        node_comm.Barrier()
        slot = rank * stride
        t0 = time.perf_counter()
        host_shm_accumulate(view, slot, SHM_ITERS)
        dt_s = time.perf_counter() - t0
        node_comm.Barrier()
        t_max = node_comm.allreduce(dt_s, op=MPI.MAX)
        t_all = node_comm.gather(dt_s, root=0)
        ok = int(view[slot]) == SHM_ITERS
        all_ok = node_comm.allreduce(int(ok), op=MPI.MIN) == 1
        lines = sorted({(ctx["shm_aligned_addr"] + 8 * r * stride) // 64 -
                        ctx["shm_aligned_addr"] // 64 for r in range(nsz)})
        results[stride] = dict(
            stride_words=stride, stride_bytes=8 * stride,
            max_rank_seconds=t_max,
            per_rank_seconds=t_all if rank == 0 else None,
            ns_per_update=t_max / SHM_ITERS * 1e9,
            distinct_cpu_lines=len(lines), cpu_line_indices=lines,
            counters_correct=all_ok,
        )
        node_comm.Barrier()
    base = results[8]["max_rank_seconds"]
    out = {
        "stride_1_seconds": results[1]["max_rank_seconds"],
        "stride_8_seconds": results[8]["max_rank_seconds"],
        "stride_16_seconds": results[16]["max_rank_seconds"],
        "false_sharing_slowdown_s1_over_s8": results[1]["max_rank_seconds"] / base,
        "s16_over_s8_ratio": results[16]["max_rank_seconds"] / base,
        "details": {str(k): v for k, v in results.items()},
    }
    return out


# ==============================================================================
# Audits
# ==============================================================================
def _as_text(obj):
    if isinstance(obj, dict):
        return "\n".join(str(v) for v in obj.values())
    return str(obj)


def ptx_audit():
    try:
        ptx = _as_text(tiled_stencil_kernel.inspect_asm())
    except Exception as e:  # pragma: no cover
        return {"error": repr(e)}
    with open("tiled_stencil_kernel.ptx", "w", encoding="utf-8") as f:
        f.write(ptx)
    bar = ptx.count("bar.sync")
    barrier = ptx.count("barrier.sync")
    return {
        "st.shared.f64": ptx.count("st.shared.f64"),
        "ld.shared.f64": ptx.count("ld.shared.f64"),
        "bar.sync": bar + barrier,
        "bar.sync_literal": bar,
        "barrier.sync_literal": barrier,
        "ld.global.f64": ptx.count("ld.global.f64"),
        "st.global.f64": ptx.count("st.global.f64"),
        "ptx_lines": ptx.count("\n"),
    }


def llvm_audit():
    try:
        ir = _as_text(host_shm_accumulate.inspect_llvm())
    except Exception as e:  # pragma: no cover
        return {"error": repr(e)}
    with open("host_shm_accumulate.ll", "w", encoding="utf-8") as f:
        f.write(ir)
    try:
        asm = _as_text(host_shm_accumulate.inspect_asm())
        with open("host_shm_accumulate.s", "w", encoding="utf-8") as f:
            f.write(asm)
    except Exception:
        asm = ""
    la = ir.count("load atomic")
    sa = ir.count("store atomic")
    return {
        "load_atomic_monotonic": ir.count("load atomic i64"),
        "store_atomic_monotonic": ir.count("store atomic i64"),
        "load_atomic_total": la,
        "store_atomic_total": sa,
        "monotonic_tokens": ir.count("monotonic"),
        "atomicrmw": ir.count("atomicrmw"),
        "nogil": True,
        "register_hoisting_defeated": bool(la >= 1 and sa >= 1),
        "ir_lines": ir.count("\n"),
    }


# ==============================================================================
# TASK 5: main
# ==============================================================================
def main():
    github_user = sys.argv[1] if len(sys.argv) > 1 else os.environ.get("GITHUB_USERNAME", "")
    if not github_user or github_user.startswith("REPLACE"):
        raise SystemExit("Pass your GitHub username as argv[1].")

    ctx = init_hybrid_context(github_user)
    rank = ctx["rank"]
    cart = ctx["cart_comm"]

    log0(ctx, "=" * 78)
    log0(ctx, f"Hybrid MPI-CUDA & Cache-Line Coherency Lab | {STUDENT_NAME} ({STUDENT_ID})")
    log0(ctx, f"GitHub user      : {ctx['github_user']}")
    log0(ctx, f"GPU              : {get_gpu_name()} | {ctx['gpu_uuid']}")
    log0(ctx, f"CPU signature    : {ctx['cpu_sig']} | coherency line {ctx['line_bytes']} B")
    log0(ctx, f"INSTANCE_SEED    : {ctx['instance_seed']} (0x{ctx['instance_seed']:016x})")
    log0(ctx, "=" * 78)

    topo = cart.gather(dict(rank=rank, coords=ctx["coords"], up=ctx["up"], down=ctx["down"],
                            left=ctx["left"], right=ctx["right"]), root=0)
    if rank == 0:
        print("[Task 1] 2x2 Cartesian topology (PROC_NULL = %d):" % MPI.PROC_NULL)
        for t in topo:
            print(f"  rank {t['rank']} coords={t['coords']} up={t['up']} down={t['down']} "
                  f"left={t['left']} right={t['right']}")
        print(f"[Task 1] node_comm size={ctx['node_size']} | shm base=0x{ctx['shm_base_addr']:x} "
              f"aligned=0x{ctx['shm_aligned_addr']:x} (offset {ctx['shm_offset_words']} words, "
              f"addr%64={ctx['shm_aligned_addr'] % 64})", flush=True)

    # ---------------- Task 2/3: stencil -----------------------------------
    fields = initialize_fields(ctx)
    log0(ctx, f"[Task 2] local arrays {fields['u0'].shape} float64 | h = 1/513 = {fields['h']:.10e}")
    log0(ctx, f"[Task 2] kappa_max = {fields['kappa_max']:.12f} | dt_max = h^2/(4 kappa_max) = "
              f"{fields['dt_max']:.6e} | dt = {fields['dt']:.6e} (CFL ratio {fields['dt'] / fields['dt_max']:.3f})")

    state = GPUState(ctx, fields)
    # JIT warm-up (not timed)
    state.step_serialized()
    state.step_overlapped()
    cuda.synchronize()
    cart.Barrier()

    u_ser, e_ser, t_ser = run_mode(state, "serialized", STENCIL_STEPS)
    u_ovl, e_ovl, t_ovl = run_mode(state, "overlapped", STENCIL_STEPS)

    local_diff = float(np.max(np.abs(u_ser[1:-1, 1:-1] - u_ovl[1:-1, 1:-1])))
    max_diff = cart.allreduce(local_diff, op=MPI.MAX)
    per_rank_diff = cart.gather(local_diff, root=0)
    parity_ok = max_diff < 1e-12
    energy_rel = max(abs(e_ser[s] - e_ovl[s]) / max(abs(e_ser[s]), 1e-300) for s in ENERGY_STEPS)
    energy_parity_ok = energy_rel < 1e-12
    finite_ok = all(math.isfinite(e_ovl[s]) for s in ENERGY_STEPS)
    monotone = all(e_ovl[ENERGY_STEPS[k + 1]] <= e_ovl[ENERGY_STEPS[k]]
                   for k in range(len(ENERGY_STEPS) - 1))

    if rank == 0:
        print(f"[Task 3] gpu_serialized_ms = {t_ser:.3f} | gpu_overlapped_ms = {t_ovl:.3f} | "
              f"speedup = {t_ser / t_ovl:.4f}x")
        print(f"[Task 5] per-rank max|u_ser-u_ovl| = {per_rank_diff}")
        print(f"[Task 5] global max_diff = {max_diff:.3e} -> {'PASS' if parity_ok else 'FAIL'} (< 1e-12)")
        for s in ENERGY_STEPS:
            print(f"[Task 5] step {s:3d}: E_serialized = {e_ser[s]:.15e} | E_overlapped = {e_ovl[s]:.15e}")
        print(f"[Task 5] energy parity rel = {energy_rel:.3e} -> {'PASS' if energy_parity_ok else 'FAIL'} | "
              f"monotone dissipation = {monotone}", flush=True)

    # ---------------- Task 4: shared-memory false sharing -----------------
    shm = benchmark_shm_strides(ctx)
    if rank == 0:
        for k in ("1", "8", "16"):
            d = shm["details"][k]
            print(f"[Task 4] stride={d['stride_words']:2d} ({d['stride_bytes']:3d} B): "
                  f"{d['max_rank_seconds'] * 1e3:10.3f} ms | {d['ns_per_update']:7.3f} ns/update | "
                  f"CPU lines used={d['distinct_cpu_lines']} {d['cpu_line_indices']} | "
                  f"per-rank s={['%.4f' % x for x in d['per_rank_seconds']]} | "
                  f"counters ok={d['counters_correct']}")
        print(f"[Task 4] false-sharing slowdown stride1/stride8 = {shm['false_sharing_slowdown_s1_over_s8']:.3f}x | "
              f"stride16/stride8 = {shm['s16_over_s8_ratio']:.3f}x", flush=True)

    # ---------------- Audits ----------------------------------------------
    if rank == 0:
        ptx = ptx_audit()
        llvm = llvm_audit()
        print(f"[Audit] PTX tiled_stencil_kernel: {json.dumps(ptx)}")
        print(f"[Audit] LLVM host_shm_accumulate: {json.dumps(llvm)}", flush=True)

        manifest = {
            "student": {"name": STUDENT_NAME, "id": STUDENT_ID, "github_username": ctx["github_user"]},
            "instructor": "Sufyan bin Uzayr",
            "environment": {
                "gpu_name": get_gpu_name(), "gpu_uuid": ctx["gpu_uuid"],
                "cpu_signature": ctx["cpu_sig"], "cpu_coherency_line_bytes": ctx["line_bytes"],
                "python": platform.python_version(), "mpi_vendor": list(map(str, MPI.get_vendor())),
                "numba": __import__("numba").__version__, "numpy": np.__version__,
            },
            "instance_seed": ctx["instance_seed"],
            "instance_seed_hex": f"0x{ctx['instance_seed']:016x}",
            "topology": topo,
            "grid": {"global": [GLOBAL_NY, GLOBAL_NX], "local_interior": [LOCAL_N, LOCAL_N],
                     "local_padded": [PAD_N, PAD_N], "tile": [TILE_DIM, TILE_DIM],
                     "shared_tile": [SHARED_DIM, SHARED_DIM]},
            "physics": {"dx": fields["h"], "kappa_max": fields["kappa_max"],
                        "dt": fields["dt"], "dt_max": fields["dt_max"],
                        "cfl_ratio": fields["dt"] / fields["dt_max"],
                        "cfl_satisfied": fields["dt"] <= fields["dt_max"],
                        "lambda_damp": LAMBDA_DAMP, "steps": STENCIL_STEPS,
                        "init_params": fields["init_params"]},
            "timing": {"gpu_serialized_ms": t_ser, "gpu_overlapped_ms": t_ovl,
                       "overlap_speedup": t_ser / t_ovl},
            "verification": {"max_diff": max_diff, "per_rank_max_diff": per_rank_diff,
                             "parity_pass": parity_ok,
                             "energy_serialized": {str(k): v for k, v in e_ser.items()},
                             "energy_overlapped": {str(k): v for k, v in e_ovl.items()},
                             "energy_parity_rel": energy_rel,
                             "energy_parity_pass": energy_parity_ok,
                             "energy_finite": finite_ok,
                             "energy_monotone_dissipation": monotone},
            "shm_window": {"slots": SHM_SLOTS, "aligned_addr_mod_64": ctx["shm_aligned_addr"] % 64,
                           "iters_per_rank": SHM_ITERS, **shm},
            "ptx_audit": ptx,
            "llvm_audit": llvm,
        }
        with open("hybrid_lab_manifest.json", "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2, default=str)
        print("[Task 5] wrote hybrid_lab_manifest.json")
        status = parity_ok and energy_parity_ok and finite_ok
        print(f"[Task 5] OVERALL: {'PASS' if status else 'FAIL'}", flush=True)

    cart.Barrier()
    ctx["win"].Free()


if __name__ == "__main__":
    main()


Writing hybrid_mpi_cuda_cache.py


## Cell 4 — Multi-rank hybrid execution

In [5]:
import os, subprocess
env = dict(os.environ, OMPI_ALLOW_RUN_AS_ROOT="1", OMPI_ALLOW_RUN_AS_ROOT_CONFIRM="1")
cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "--bind-to", "none",
    "--mca", "btl_vader_single_copy_mechanism", "none",
    "-np", "4",
    "python3",
    "hybrid_mpi_cuda_cache.py",
    GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True, env=env)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Hybrid MPI-CUDA execution failed with code {proc.returncode}")


Hybrid MPI-CUDA & Cache-Line Coherency Lab | Ainat Yermurat (230103161)
GitHub user      : yermy-bc
GPU              : Tesla T4 | GPU-501dd0b0-086c-7321-b66a-8e2cf85952c5
CPU signature    : x86_64::2::Intel(R) Xeon(R) CPU @ 2.00GHz | coherency line 64 B
INSTANCE_SEED    : 2272905753577972645 (0x1f8afbbd91f4e3a5)
[Task 1] 2x2 Cartesian topology (PROC_NULL = -2):
  rank 0 coords=(0, 0) up=-2 down=2 left=-2 right=1
  rank 1 coords=(0, 1) up=-2 down=3 left=0 right=-2
  rank 2 coords=(1, 0) up=0 down=-2 left=-2 right=3
  rank 3 coords=(1, 1) up=1 down=-2 left=2 right=-2
[Task 1] node_comm size=4 | shm base=0x7f8e65301108 aligned=0x7f8e65301140 (offset 7 words, addr%64=0)
[Task 2] local arrays (258, 258) float64 | h = 1/513 = 1.9493177388e-03
[Task 2] kappa_max = 0.899987745653 | dt_max = h^2/(4 kappa_max) = 1.055525e-06 | dt = 8.444203e-07 (CFL ratio 0.800)
[Task 3] gpu_serialized_ms = 720.057 | gpu_overlapped_ms = 995.676 | speedup = 0.7232x
[Task 5] per-rank max|u_ser-u_ovl| = [0.0, 0.0, 

## Cell 5 — Manifest & audit excerpts

In [6]:
import json
m = json.load(open("hybrid_lab_manifest.json"))
print(json.dumps({k: m[k] for k in ["student", "instance_seed_hex", "physics", "timing", "verification", "ptx_audit", "llvm_audit"]}, indent=2))
print("SHM:", json.dumps({k: v for k, v in m["shm_window"].items() if k != "details"}, indent=2))
!grep -nE "bar.sync|barrier.sync" tiled_stencil_kernel.ptx | head
!grep -nE "atomic" host_shm_accumulate.ll | head


{
  "student": {
    "name": "Ainat Yermurat",
    "id": "230103161",
    "github_username": "yermy-bc"
  },
  "instance_seed_hex": "0x1f8afbbd91f4e3a5",
  "physics": {
    "dx": 0.001949317738791423,
    "kappa_max": 0.8999877456529193,
    "dt": 8.444203079699078e-07,
    "dt_max": 1.0555253849623847e-06,
    "cfl_ratio": 0.8,
    "cfl_satisfied": true,
    "lambda_damp": 0.15,
    "steps": 300,
    "init_params": {
      "x0": 0.48299396114236837,
      "y0": 0.6273197175082164,
      "amp": 0.9037633843488584,
      "sigma": 0.09781022604340775,
      "phase": 5.295413969629746
    }
  },
  "timing": {
    "gpu_serialized_ms": 720.0566220001292,
    "gpu_overlapped_ms": 995.675649998816,
    "overlap_speedup": 0.7231839223957978
  },
  "verification": {
    "max_diff": 0.0,
    "per_rank_max_diff": [
      0.0,
      0.0,
      0.0,
      0.0
    ],
    "parity_pass": true,
    "energy_serialized": {
      "0": 0.01227433210203259,
      "100": 0.012204064399528645,
      "200": 0.

### Analytical & Forensic Engineering Responses

*Ainat Yermurat (230103161). All numbers are from the run above: Colab Tesla T4 (`GPU-501dd0b0-…`), Intel Xeon @ 2.00 GHz with 2 vCPUs, 64 B coherency line, `INSTANCE_SEED = 0x1f8afbbd91f4e3a5`.*

- **Question 1 (GPU Coalescing vs. CPU Cache-Line Granularity):**
  Each padded row is 258 × 8 B = **2,064 B**, so in the column `u[1:257, 1]` consecutive elements sit 2,064 B apart. That is far more than one 64 B CPU line, one 32 B GPU sector or one 128 B L1 line.
  *CPU:* every element lands on its own 64 B line, so one column costs 256 lines × 64 B = **16,384 B** of traffic to deliver 2,048 B of useful data (12.5 % efficiency). The stride prefetcher can hide the latency, but it cannot reduce the bytes moved.
  *T4 GPU:* a warp reading 32 elements of that column produces 32 separate 32 B sector requests instead of 2 coalesced 128 B transactions. One column costs 256 × 32 B = **8,192 B** of L2/DRAM traffic for 2,048 B useful (25 % efficiency, 6,144 B wasted). If it is served through 128 B L1 lines, the cost is 32,768 B (6.25 %).
  *PCIe:* without packing there are two bad choices. One is 256 tiny strided `cudaMemcpy` calls per column, each paying about 10 µs of launch latency. The other is copying the whole padded tile, 258² × 8 = **532,512 B** per rank per step (159.75 MB over 300 steps). `pack_halos_kernel` gathers the 2 strided columns and 2 contiguous rows into one contiguous 1,024-double buffer of **8,192 B**, which goes as a single pinned D2H DMA (2.46 MB over 300 steps). That is **65× less PCIe traffic** than the full-tile copy. The only strided accesses left happen once, on the GPU, where the HBM/GDDR6 bandwidth (~320 GB/s) is far cheaper than PCIe (~12 GB/s effective).

- **Question 2 (Shared-Memory Tile Synchronization & PTX Audit):**
  PTX audit of `tiled_stencil_kernel.inspect_asm()`: **`st.shared.f64` = 6, `ld.shared.f64` = 10, `bar.sync` = 1** (a single `bar.sync 0;` at PTX line 373). For comparison, `ld.global.f64` = 7 and `st.global.f64` = 1.
  - The 10 shared loads are the 5-point stencil read twice: centre plus N/S/E/W for `s_u`, and the same five for `s_k`.
  - The 6 shared stores come from the cooperative 18 × 18 fill of the two tiles (load-or-zero for `s_u` and `s_k`). The compiler restructured the loop: every thread runs at least one iteration because `tid < 256 < 324`.
  - The one `bar.sync` is the single `cuda.syncthreads()` that separates the fill phase from the compute phase.
  - There is exactly one `st.global.f64` (the `u_new[i, j]` write). Global loads only fill the tile, plus one read of `source[i, j]`.

  A 16 × 16 block has 256 threads, which is **8 warps** (`tid = 32w … 32w+31`). Shared cell `idx = 18·r + c` is written by thread `idx` if `idx < 256`, otherwise by thread `idx − 256` on the second loop iteration. Thread `(ty, tx)` reads tile rows `ty … ty+2`.
  - Example: thread `(15, tx)` in warp 7 reads rows 16–17 (`idx` 288–323). Those cells are written on the *second* iteration by threads 32–67, which belong to **warps 1–2**.
  - Likewise, thread `(1, tx)` (warp 0) reads row 3 (`idx` 54–71), which is written by warps 1–2.

  Without the barrier the warp scheduler can let warp 7 issue its `ld.shared` before warps 1–2 have issued the matching `st.shared`. That is a cross-warp **Read-After-Write hazard**: the thread reads whatever was in shared memory before (zeros, or data from the previous block on that SM), and the hazard concentrates on the halo rows and on warp-boundary rows. The result is wrong and non-deterministic, so the `max_diff = 0.000e+00` parity check would fail. With `bar.sync 0`, no warp gets past the barrier until all 8 warps have finished their stores.

- **Question 3 (Dual-Stream Compute-Communication Overlap on Shared GPU Silicon):**
  Measured over 300 steps: **`gpu_serialized_ms` = 720.057 ms** (2.400 ms/step) and **`gpu_overlapped_ms` = 995.676 ms** (3.319 ms/step). The speedup is **0.7232×**, so the overlapped pipeline was **38 % slower**. The results were still bit-identical (`max_diff = 0`).
  Why overlap lost here:
  1. **Four contexts on one T4.** Each MPI rank creates its own CUDA context on the same GPU, and MPS is not running. The GPU time-slices between contexts, and kernels from different processes never run concurrently. `stream_interior` and `stream_boundary` can only overlap work *inside one context*. The two copy engines and the compute engine are shared by all 4 ranks and driven by the context scheduler.
  2. **Too little work to hide.** The deep interior is 254² ≈ 64.5 k points, only tens of µs on a T4, while each step's cost is dominated by host work: Python launches, `Waitall`, and two stream synchronisations. Overlapped mode issues **6 kernels** per step (pack, deep interior, unpack, 4 strips) instead of 3. The strip kernels are badly shaped: a 1 × 256 row strip uses a 16 × 1 grid of 16 × 16 blocks, so only 1/16 of the threads do work.
  3. **Oversubscribed CPU.** Four ranks share 2 vCPUs, so extra launches and synchronisations turn directly into wall-clock time.

  *With 4 dedicated GPUs and GPUDirect RDMA:* each rank owns its context, SMs and copy engines, and the InfiniBand NIC reads and writes device memory directly over PCIe peer-to-peer. The halo never goes through pinned host buffers (no D2H/H2D staging) and no context switching is needed. The interior kernel can then run truly concurrently with the halo transfer, and the dual-stream design would give a speedup above 1×.

- **Question 4 (MPI-3 Shared-Memory MESI RFO Ping-Pong):**
  Measured with 40,000,000 RMW updates per rank in the 64 B-aligned `MPI.Win.Allocate_shared` window (`addr % 64 = 0`). The LLVM IR shows `load atomic i64 … monotonic` / `store atomic i64 … monotonic` (9 each after 4× unrolling, `atomicrmw` = 0), which proves the counter is not kept in a register.

  | stride | spacing | CPU lines used | time (max rank) | ns/update |
  |---|---|---|---|---|
  | 1 | 8 B | 1 (`[0]`) | **540.430 ms** | 13.511 |
  | 8 | 64 B | 4 (`[0,1,2,3]`) | **569.468 ms** | 14.237 |
  | 16 | 128 B | 4 (`[0,2,4,6]`) | **257.950 ms** | 6.449 |

  **False-sharing slowdown (stride 1 / stride 8) = 0.949×**, so no penalty was measured on this machine. The cause is the VM topology: `lscpu` reports 2 vCPUs with **L1d 32 KiB (1 instance)** and **L2 1 MiB (1 instance)**. The two vCPUs are SMT siblings of a *single physical core*, sharing one L1D. All 4 ranks are time-sliced onto those 2 hardware threads, so the line holding `shm_view[0..7]` simply stays in **M** in the one shared L1 and never has to cross a core-to-core coherence link.

  MESI trace on real multi-core hardware (Rank 0 on core A, Rank 1 on core B, both slots on line L0):
  1. A loads `shm_view[0]`: miss, no other copy, so A gets L0 in **I→E**.
  2. A stores: silent **E→M** (no bus traffic).
  3. B loads `shm_view[1]`: miss. The snoop hits A in M, so A forwards and writes back the line, **A: M→S**, **B: I→S**.
  4. B stores: B issues an RFO/upgrade-invalidate, **A: S→I**, **B: S→M**.
  5. A loads `shm_view[0]` again: miss (I). The snoop hits B in M, **B: M→S**, **A: I→S**.
  6. A stores: RFO, **B: S→I**, **A: S→M**. Then the cycle repeats from step 3.

  Every iteration therefore pays a cross-core line transfer (~40–100 ns on Xeon), compared with ~1–2 ns when each rank owns its line. With 4 ranks the line bounces among four cores. On this single-core SMT VM those transitions never happen, which is why the stride-1 and stride-8 timings are equal (13.5 vs 14.2 ns/update).

- **Question 5 (128-Byte Spatial Prefetcher Pairing & Padding Trade-Offs):**
  Measured: **stride 8 (64 B) = 569.468 ms**, **stride 16 (128 B) = 257.950 ms**. Ratio 16/8 = **0.453×**, so 128 B padding was **2.21× faster**.
  Intel's L2 *adjacent-cache-line* (spatial) prefetcher treats lines as 128 B-aligned even/odd buddy pairs: a miss on line 2k pulls line 2k+1 too, and the reverse. With stride 8, the four counters sit in lines `[0,1,2,3]`, so ranks 0/1 share pair 0 and ranks 2/3 share pair 1. A miss or RFO by one rank makes the prefetcher fetch its neighbour's line, and on a multi-core system it steals that line back. The result is still ping-pong at pair granularity even though the 64 B lines are technically distinct. With stride 16, the counters sit in lines `[0,2,4,6]`, so each rank owns a whole 128 B pair and the prefetcher only ever brings in an untouched padding line.
  Caveat: on this 1-core SMT VM there is no cross-core coherence, so not all of the 2.2× gap can be credited to the prefetcher. Part of it is SMT memory-disambiguation and store-buffer interaction between the sibling threads on nearby lines, and part is scheduler noise: per-rank times for stride 16 were 0.2579 / 0.1879 / 0.2525 / 0.2545 s, compared with a tight 0.562–0.570 s band for stride 8.
  *Trade-off:* padding to 128 B costs 16× the memory of packed 8 B counters. Four counters cost only 512 B, but per-thread or per-bin arrays grow quickly. This core's 32 KiB L1d holds just 512 lines, so it fits only 256 counters at 128 B padding. 8,192 padded counters already fill the whole **1 MiB L2**, after which useful data gets evicted. Padding also wastes DRAM bandwidth and TLB entries on dead bytes. The usual rule is to pad to 128 B only for a small number of heavily written, cross-core counters, and keep read-mostly data packed.

- **Question 6 (CFL Stability & Distributed Energy Dissipation):**
  From the manifest: **κ_max = 0.8999877456529193** (global `allreduce(MAX)`), Δx = 1/513 = 1.949317738791423 × 10⁻³, Δx² = 3.799840 × 10⁻⁶.
  Δt_max = Δx² / (4·κ_max) = 3.799840 × 10⁻⁶ / 3.599951 = **1.055525 × 10⁻⁶**.
  Simulation Δt = **8.444203 × 10⁻⁷** = 0.800 · Δt_max (`cfl_satisfied: true`), so the Courant condition holds with a 20 % margin. The damping term barely changes this: the combined limit 1/(4κ_max/Δx² + λ/2) = 1.0555253 × 10⁻⁶ differs only in the 8th digit, because λΔt = 1.27 × 10⁻⁷. The run covers 300·Δt = 2.533 × 10⁻⁴ time units.
  Global energy E = ½ Δx² Σ u², summed with `cart_comm.allreduce(SUM)` with no gather. The serialized and overlapped values are identical (relative difference 0.000e+00):

  | step | E (global) | ΔE vs previous | relative drop |
  |---|---|---|---|
  | 0 | 1.227433210203259e-02 | — | — |
  | 100 | 1.220406439952865e-02 | −7.0268e-05 | 0.5725 % |
  | 200 | 1.213462827355494e-02 | −6.9436e-05 | 0.5690 % |
  | 300 | 1.206600684391630e-02 | −6.8621e-05 | 0.5655 % |

  The energy decreases monotonically (`energy_monotone_dissipation: true`), by **1.697 % in total**, at an almost constant effective rate of −d ln E/dt ≈ 67.6 s⁻¹. This matches the energy balance dE/dt = −∫κ|∇u|² − λ∫u² + ∫S·u: diffusion of the seed-dependent Gaussian (σ = 0.0978, amplitude 0.904) dominates the source term. The decay rate shrinks slightly each window (0.5725 → 0.5690 → 0.5655 %) because the Gaussian spreads and its gradients weaken. No oscillation or growth appears anywhere, which confirms the explicit scheme is stable.
